# Mechanosorption
<strong>J. M. Maas @ IfB, ETH Zürich, v. 2026-08-12</strong>

<div class='alert alert-info'>
    Select ▸▸ <strong>Restart the kernel and run all cells</strong> to run the widgets in this notebook and explore the mechanosorptive data.
</div>

In [ ]:
# Import required libraries
try:
    import odf
except:
    !pip3 install highlight-text odfpy
    
import numpy as np
import pandas as pd
import scipy as sp

import matplotlib as mpl
import matplotlib.pyplot as plt
import ipywidgets as widgets
%matplotlib widget

from pathlib import Path
from datetime import datetime


# Required modules
import utils.rheonomic_utils as rhutils

The next cell imports the data points.

In [ ]:
# Import data points

# Import experimental strains
file_strains_avg = Path('data/06_mechanosorption') / 'experimental_strains_average.csv'
file_strains = Path('data/06_mechanosorption') / 'experimental_strains_samples.csv'

df_strains_avg = pd.read_csv(file_strains_avg, index_col=['sample_type', 'field', 'times']).sort_index()
df_strains = pd.read_csv(file_strains, index_col=['sample_type', 'field', 'sample', 'times']).sort_index()

# Import model response from Abaqus
files_model = list(Path('data/06_mechanosorption/simulation_strains').glob('simulation_result_*.csv'))
df_model = pd.concat(
    [pd.read_csv(f, index_col=['time']) for f in files_model],
    keys=[f.stem.split('_')[-1] for f in files_model],
    names=['sample_type', 'times']
).sort_index()

# Import strains from inplane movements due to hygroexpansion
files_inplane = list(Path('data/06_mechanosorption/simulation_strains').glob('inplane_correction_*.csv'))
df_inplane = pd.concat(
    [pd.read_csv(f, index_col=['field', 'times']) for f in files_inplane],
    keys=[f.stem.split('_')[-1] for f in files_inplane],
    names=['sample_type', 'field', 'times']
).sort_index()


# Import model coefficients
file_phi = Path('data/06_mechanosorption') / 'kelvin_voigt_compliances.csv'
df_phi = pd.read_csv(file_phi, index_col=[0])
df_phi.columns = ['phi1', 'phi2', 'phi3']


# Import climate boundary conditions
files_climate = list(Path('data/06_mechanosorption/boundary_conditions/climate').glob('abaqus_mc_AVG_*.csv'))
df_climate = pd.concat(
    [pd.read_csv(f, index_col='Time') for f in files_climate],
    keys=[f.stem.split('_')[-1] for f in files_climate],
    names=['sample_type', 'times']
).sort_index()

files_climate_all = list(Path('data/06_mechanosorption/boundary_conditions/climate').glob('abaqus_mc_MACRO_*.csv'))
df_climate_all = pd.concat(
    [pd.read_csv(f, index_col='Time') for f in files_climate_all],
    keys=[f.stem.split('_mc_')[-1] for f in files_climate_all],
    names=['experiment', 'times']
).sort_index()

# Import load boundary conditions
files_load = list(Path('data/06_mechanosorption/boundary_conditions/load').glob('abaqus_load_AVG_*.csv'))
df_load = pd.concat(
    [pd.read_csv(f, index_col='Time') for f in files_load],
    keys=[f.stem.split('_')[-1] for f in files_load],
    names=['sample_type', 'times']
).sort_index()


# Import meta information
file_meta = Path('data/06_mechanosorption') / 'meta_information.ods'
df_meta = rhutils.read_meta(file_meta, [f'MACRO_MS_RH35_RH90_{i+1:02d}' for i in range(11)])
sample2experiment = pd.Series(data=df_meta.index.get_level_values('experiment'), index=pd.Index(df_meta['name'], name='sample'))

The next cell visualizes the strain data for a selected anatomical direction and allows for exporting the data points and mechanosorptive parameters of the shown graph.

In [ ]:
# Define mechanosorption visualization

# Initialize export buffer
buffer_strain = [pd.DataFrame()]
buffer_mc = [pd.DataFrame()]
buffer_load = [pd.DataFrame()]
buffer_coefficients = [pd.DataFrame()]

# Function for extracting strains from model response
def extract_model_strains(df, sample_type, field):
    '''
    Extract the strain components from a simulation response.$

    Args:
        df (pd.DataFrame): Dataframe containing columns 'SDVi' and index
            with times from simulation response.
        sample_type (str): Indexes of SDVi are chosen based on this sample
            type
        field (str): Indexes of SDVi are chosen base on this field.
            Options: 'exx', 'exy', 'eyy'.

    Returns:
        pd.DataFrame: Selected strain components and total strain.
    '''
    # Select correct index
    match field:
        case 'eyy':
            s2i = dict(cR=1, tR=1, cT=2, tT=2, cLx=3, tLx=3)
        case 'exx':
            s2i = dict(cR=3, tR=3, cT=1, tT=1, cLx=2, tLx=2)
        case 'exy':
            s2i = dict(sRT=4, sTR=4, sRL=5, sLR=5, sTL=6, sLT=6)
        case _:
            raise ValueError(f'Invalid field {field}.')
    i = s2i[sample_type] - 1

    # Select data from dataframe
    i2col = dict(el=f'SDV{2+i}', pl=f'SDV{8+i}', ve=f'SDV{59+i}', ms=f'SDV{65+i}', hyg=f'SDV{71+i}')
    df_export = df[i2col.values()].copy()
    df_export.columns = i2col.keys()
    df_export['tot'] = df_export.sum(axis=1)

    if 's' in sample_type:
        df_export['tot'] *= np.sign(np.median(df_export['tot']))

    return df_export


# Remove numerical artifacts
def smooth_artifacts(series):
    '''
    Remove numerical artifacts from series

    Args:
        series (pd.Series): Index resembles time and data resembles strains.

    Returns:
        pd.Series: Series, with artifacts removed.
    '''
    # Initialize
    x = series.copy()

    # Identify peaks
    peaks, _ = sp.signal.find_peaks(x, distance=32)
    npeaks, _ = sp.signal.find_peaks(-x, distance=32)

    # Select indexes for smoothing
    rm_idx = np.concatenate([peaks, npeaks])
    new_rm_idx = []
    for i in rm_idx:
        new_rm_idx.append(i)
        new_rm_idx.append(i-1)
        new_rm_idx.append(i+1)
    rm_idx = list(set(new_rm_idx))

    # Remove artifacts
    x.iloc[rm_idx] = np.nan
    x = x.ffill()
    return x


# Mechanosorptive strain plotting function
def plot_ms_strains(sample_types, field, ax):
    # Initialize
    sf2label = dict(
        exx=dict(cR='c,RL', tR='t,RL', cT='c,TR', tT='t,TR', cLx='c,LT', tLx='t,LT'),
        exy=dict(sRT='s,RT', sTR='s,TR', sRL='s,RL', sLR='s,LR', sTL='s,TL', sLT='s,LT'),
        eyy=dict(cR='c,RR', tR='t,RR', cT='c,TT', tT='t,TT', cLx='c,LL', tLx='t,LL')
    )
    inplane_corr = True
    
    # Plot strains
    colors = ['cornflowerblue', 'chocolate', 'green']
    model_colors = ['#040273', '#610023']
    model_ls = ['-.', '--']
    n, sigma = [], []
    dfs, keys = [], []
    for i, sample_type in enumerate(sample_types):

        # Select inplane strain
        if inplane_corr and not 's' in sample_type:
            strain_inplane = df_inplane.loc[(sample_type, field), 'corr']
            inplane_fun = sp.interpolate.interp1d(
                strain_inplane.index,
                strain_inplane,
                fill_value=(strain_inplane.iloc[0], strain_inplane.iloc[-1]),
                bounds_error=False
            )
        else:
            inplane_fun = lambda x: 0.0
        
        # Select strain data
        data_strain_avg = df_strains_avg.loc[(sample_type, field)]
        data_strain_samples = df_strains.loc[(sample_type, field)]
        samples = data_strain_samples.index.get_level_values('sample').unique()
    
        # Select stress
        idxmax = data_strain_avg['stress'].abs().idxmax()
        stress_max = data_strain_avg.loc[idxmax, 'stress']
        sigma.append(stress_max)
    
        # Plot experimental strain data
        times = data_strain_avg.index
        strain = data_strain_avg['strain'] - inplane_fun(times)
        label = fr'$\epsilon_{{{sf2label[field][sample_type]}}}^\mathrm{{tot}}\ (\sigma = {stress_max:.2f}\,\mathrm{{MPa}})$'
        label = fr'$\epsilon_{{{sf2label[field][sample_type]}}}^\mathrm{{tot}}\ (n = {len(samples)})$'
        ax.plot(times, strain, label=label, c=colors[i], lw=2.0)
        dfs.append(pd.DataFrame(data={'strain': strain}, index=pd.Index(times, name='times'))), keys.append(f'average_{sample_type}')
    
        # Plot samples
        for sample in samples:
            data_sample = data_strain_samples.loc[sample]
            sample_times = data_sample.index
            sample_strain = data_sample['strain'] - inplane_fun(sample_times)
            ax.plot(sample_times, sample_strain, zorder=1, lw=1.0, ls=':', c=colors[i])
            dfs.append(pd.DataFrame(data={'strain': sample_strain}, index=pd.Index(sample_times, name='times'))), keys.append(sample)
        n.append(len(samples))
    
        # Plot model
        if sample_type in df_model.index.get_level_values('sample_type').unique():
            data_model = extract_model_strains(df_model.loc[sample_type], sample_type, field)
            times_model = data_model.index
            strains_model = smooth_artifacts(data_model['tot']) if (sample_type == 'cR') or (sample_type == 'tR') else data_model['tot']
            ax.plot(times_model, strains_model, c=model_colors[i], ls=model_ls[i], lw=1.2, zorder=5)
            dfs.append(pd.DataFrame(data={'strain': strains_model}, index=pd.Index(times_model, name='times'))), keys.append(f'model_{sample_type}')
    
        # Plot climate
        data_climate = df_climate.loc[sample_type]
        cl_times = data_climate.index
        cl_mc = data_climate['TEMP']
        cl_c = 'k'
    
        # Mark loaded area
        unload_time = data_strain_avg.query('state == "unload"').index[-1]
        ax.axvspan(0, unload_time, facecolor='#cccccc', zorder=-3)

        # Plot moist sections as background color
        condition = data_climate['TEMP'] > 10
        group_ids = (condition != condition.shift()).cumsum()
        connected_sections = data_climate[condition].groupby(group_ids)
        for _, section in connected_sections:
            start_time = section.index[0]
            end_time = section.index[-1]
            ax.axvspan(start_time, end_time, color='steelblue', alpha=0.1, zorder=-2)

    # Format plot
    ax.legend(loc='upper right')
    ax.axhline(c='gray', ls='-', lw=1.0, zorder=-1)
    ax.grid(True)
    ax.set_xlim([np.min(times), np.max(times)])
    ax.set_ylabel(r'$\epsilon^\mathrm{tot}$ [-]')

    # Store buffer
    data_buffer = pd.concat(dfs, keys=keys, names=['sample', 'times'])
    buffer_strain[0] = data_buffer.sort_index()


# Moisture plotting function
def plot_moisture(sample_types, field, ax):
    # Initialize
    colors = ['cornflowerblue', 'chocolate', 'green']
    model_colors = ['#040273', '#610023']
    model_ls = ['-.', '--']

    # Iterate over sample types
    dfs, keys = [], []
    for i, sample_type in enumerate(sample_types):

        # Identify samples
        data_strain_samples = df_strains.loc[(sample_type, field)]
        samples = data_strain_samples.index.get_level_values('sample').unique()

        # Plot average climate data
        data_climate = df_climate.loc[sample_type]
        ax.plot(data_climate.index, data_climate, c=colors[i], lw=2.0)
        dfs.append(data_climate), keys.append(f'average_{sample_type}')

        # Plot moist sections as background color
        condition = data_climate['TEMP'] > 10
        group_ids = (condition != condition.shift()).cumsum()
        connected_sections = data_climate[condition].groupby(group_ids)
        for _, section in connected_sections:
            start_time = section.index[0]
            end_time = section.index[-1]
            ax.axvspan(start_time, end_time, color='steelblue', alpha=0.1, zorder=-2)

        # Plot individual climate data
        for sample in samples:
            data_mc = df_climate_all.loc[sample2experiment[sample]]
            ax.plot(data_mc.index, data_mc['TEMP'], zorder=1, lw=1.0, ls=':', c=colors[i])
            dfs.append(data_mc), keys.append(sample)

    # Format plot
    ax.grid(True)
    ax.set_ylabel(r'$\omega_\mathrm{eq}$ [%]')

    # Store buffer
    data_buffer = pd.concat(dfs, keys=keys, names=['sample', 'times'])
    data_buffer.columns = ['mc']
    buffer_mc[0] = data_buffer.sort_index()


# Load plotting function
def plot_load(sample_types, field, ax):
    # Initialize
    colors = ['cornflowerblue', 'chocolate', 'green']
    model_colors = ['#040273', '#610023']
    model_ls = ['-.', '--']

    # Iterate over sample types
    dfs, keys = [], []
    for i, sample_type in enumerate(sample_types):

        # Identify samples
        data_strain_samples = df_strains.loc[(sample_type, field)]
        samples = data_strain_samples.index.get_level_values('sample').unique()

        # Plot average load
        data_load = df_load.loc[sample_type]
        force = data_load['Load']
        max_force = force.loc[force.abs().idxmax()]
        area = 4*12 if 't' in sample_type else (4*15 if 'c' in sample_type else 4*12.83)
        stress = force / area
        ax.plot(stress.index, stress, c=colors[i], lw=2.0)
        dfs.append(stress.to_frame()), keys.append(f'average_{sample_type}')

        # Plot individual stress data
        for sample in samples:
            stress = df_meta.loc[df_meta['name'] == sample]['stress'].iloc[0]
            stress = force / max_force * stress
            ax.plot(stress.index, stress, zorder=1, lw=1.0, ls=':', c=colors[i])
            dfs.append(stress.to_frame()), keys.append(sample)

        # Plot moist sections as background color
        data_climate = df_climate.loc[sample_type]
        condition = data_climate['TEMP'] > 10
        group_ids = (condition != condition.shift()).cumsum()
        connected_sections = data_climate[condition].groupby(group_ids)
        for _, section in connected_sections:
            start_time = section.index[0]
            end_time = section.index[-1]
            ax.axvspan(start_time, end_time, color='steelblue', alpha=0.1, zorder=-2)

    # Format plot
    ax.grid(True)
    ax.set_xlabel(r'$t$ [h]')
    ax.set_ylabel(r'$\sigma$ [MPa]')

    # Store buffer
    data_buffer = pd.concat(dfs, keys=keys, names=['sample', 'times'])
    data_buffer.columns = ['stress']
    buffer_load[0] = data_buffer.sort_index()


# Display coefficients
def display_coefficients(component):
    # Set identifier
    c2ids = {
        'E_RR': ('c, RR', 't, RR'),
        'E_TT': ('c, TT', 't, TT'),
        'E_LL': ('c, LL', 't, LL'),
        'G_RT': ('s, RT', 's, TR'),
        'G_RL': ('s, RL', 's, LR'),
        'G_TL': ('s, TL', 's, LT'),
        'E_RT': ('c, TR', 't, TR'),
        'E_RL': ('c, RL', 't, RL'),
        'E_TL': ('c, LT', 't, LT'),
    }
    ids = c2ids[component]

    # Display
    data = df_phi.loc[[ids[0] in i or ids[1] in i for i in df_phi.index]].copy()
    taus = [10., 50., 250.]
    for i in range(len(taus)):
        data[f'mu{i+1:d}'] = taus[i]
        
    display(data)

    # Store buffer
    buffer_coefficients[0] = data.copy()


# Widget function
def widget_function(component, fig, axs):
    # Initialize
    [ax.cla() for ax in axs]

    # Select sample types
    component2args = {
        'E_RR': (['cR', 'tR'], 'eyy'),
        'E_TT': (['cT', 'tT'], 'eyy'),
        'E_LL': (['cLx', 'tLx'], 'eyy'),
        'G_RT': (['sRT', 'sTR'], 'exy'),
        'G_RL': (['sRL', 'sLR'], 'exy'),
        'G_TL': (['sTL', 'sLT'], 'exy'),
        'E_RT': (['cT', 'tT'], 'exx'),
        'E_RL': (['cR', 'tR'], 'exx'),
        'E_TL': (['cLx', 'tLx'], 'exx'),
    }
    sample_types, field = component2args[component]

    # Display coefficients
    display_coefficients(component)

    # Create plots
    plot_ms_strains(sample_types, field, axs[0])
    plot_moisture(sample_types, field, axs[1])
    plot_load(sample_types, field, axs[2])

    # Final formatting
    fig.align_ylabels(axs)


# Export function
def export_data(df, suffix):
    component = w_component.value
    export_path = f'export_mechanosorption_{component}_{suffix}.csv'
    df.to_csv(export_path)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with w_interactive.children[-1]:
        w_interactive.children[-1].clear_output()
        print(f'({timestamp}) Successfully saved data to {export_path}.')

def export_call(button):
    export_data(buffer_strain[0], 'strain')
    export_data(buffer_mc[0], 'climate')
    export_data(buffer_load[0], 'load')
    export_data(buffer_coefficients[0], 'coefficients')
    
button_export = widgets.Button(description='Export CSV', button_style='primary', icon='save')
button_export.on_click(export_call)
display(button_export)


# Initialize plot
fig, axs = plt.subplots(3, 1, figsize=(5, 5), sharex=True, gridspec_kw=dict(height_ratios=[0.6, 0.2, 0.2], hspace=0.15, left=0.2, right=0.98, bottom=0.10, top=0.98))

# Create widget
w_component = widgets.Dropdown(options=['E_RR', 'E_TT', 'E_LL', 'G_RT', 'G_RL', 'G_TL', 'E_RT', 'E_RL', 'E_TL'])
w_interactive = widgets.interactive(
    widget_function,
    component=w_component,
    fig=widgets.fixed(fig),
    axs=widgets.fixed(axs)
)
display(w_interactive)

Use the control elements to explore the data and the button to export the data points of the currently selected graph.